# 08 -- Documentation, validation & monitoring pack

**What this notebook does (plain English):** The write-up a model-validation or
consulting team would hand over: what we built, how, what the results were, the
limitations, and how we'd **monitor** the models once live. It includes a
**stability check (PSI)** -- a standard early-warning gauge that flags when the
loans coming through the door no longer look like the ones a model was built on.

**Headline result:** the population shifts materially between the calm and crisis
books (high PSI), exactly the kind of drift monitoring is designed to catch.

## Model development summary

**Objective.** Quantify mortgage credit risk end-to-end -- PD, LGD, EAD,
Expected Loss and a downturn stress test -- on the Freddie Mac Single-Family
Loan-Level Dataset.

**Data.** 50,000-loan samples for **17 origination years (2006-2022)** -- spanning the
housing boom, the GFC, the recovery, the long expansion and COVID-2020, observed through
2025-09; origination characteristics joined to monthly performance and collapsed to one
row per loan (~850k loans). Raw data is not redistributed in this repo.

**Methodology.**
- *Default* = first month at 180+ days past due, or a credit-event zero-balance
  code (third-party sale, short sale/charge-off, REO disposition, note sale).
- *PD* = logistic regression on origination features (interpretable scorecard), on a
  **one-year** default target, calibrated to a count-weighted long-run average per grade,
  with a formal calibration test, a margin of conservatism, and the 5 bps floor
  -- see the PD framework-alignment notes below.
- *LGD* = two-stage model (P(loss) x severity) on **realised** losses from
  defaulted, disposed loans; reconciled to Freddie Mac's own loss field (corr ~0.99).
  Extended to **economic (discounted) loss**, an **APRA capital view** (MI excluded,
  20% high-LVR reduction, 20% floor), a margin of conservatism, a downturn LGD, and an
  **independent LGD validation** (notebook 04b) -- see the framework-alignment notes below.
- *EAD* = outstanding balance at default (no CCF -- a term loan has no undrawn limit).
- *EL* = PD x LGD x EAD, staged under IFRS 9 / AASB 9.
- *Stress* = downturn multipliers observed in the **GFC (2006-09)** vintages, plus a
  separate **observed COVID-2020** scenario (high default, mild severity).

**Results.** Across the cycle, GFC origination years run ~7-14% default with ~54-58% LGD
versus ~2-3% / ~20-35% in the calm expansion; COVID-2020 shows the **lowest** default
(~1.2%, forbearance-suppressed) at moderate severity. Long-run grade PDs calibrate to a
**count-weighted full-cycle average**, the downturn LGD is ~56% (GFC) vs ~34% (non-GFC),
portfolio 12-month EL is ~$236m rising to ~$323m under IFRS 9 lifetime staging, and the
severe stress lifts EL ~13x (COVID-shape ~4.7x). PD model AUC ~0.75-0.80.

**Limitations.** Portfolio demonstration, not a regulatory-capital model; US
agency mortgages, not an APRA IRB portfolio; 50k-loan samples, illustrative
calibration; macro stress is scenario-based, not a fitted macroeconomic model.

**Governance / monitoring.** Track discrimination (AUC/Gini/KS), calibration, and
**population stability (PSI)** over time; re-fit on a trigger; maintain model
documentation and an owner for each model.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and re-fit the PD model to get a score to monitor.
import pandas as pd
from src import models, metrics
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
pd_model, pd_cols = models.fit_pd(base)
base = base.copy()
base['pd_hat'] = models.predict_pd(pd_model, pd_cols, base)

In [3]:
# PSI: compare the calm reference book (expected) to the GFC crisis books (actual)
# on both a raw driver (credit score) and the model output (PD). Regime via the
# documented classifier (R3-C2).
from src import definitions as d
calm = base[base['vintage_year'] == d.CALM_REFERENCE_VINTAGE]
crisis = base[d.is_downturn_vintage(base['vintage_year'])]
psi_tbl = pd.DataFrame([
    {'feature': 'credit_score', 'psi_calm_vs_crisis': round(metrics.psi(calm['credit_score'], crisis['credit_score']), 4)},
    {'feature': 'pd_hat', 'psi_calm_vs_crisis': round(metrics.psi(calm['pd_hat'], crisis['pd_hat']), 4)},
])
psi_tbl['interpretation'] = psi_tbl['psi_calm_vs_crisis'].apply(
    lambda v: 'stable (<0.10)' if v < 0.10 else ('watch (0.10-0.25)' if v < 0.25 else 'material shift (>0.25)'))
save_csv(psi_tbl, 'outputs/tables/08_monitoring_psi.csv')
psi_tbl

,feature,psi_calm_vs_crisis,interpretation
0,credit_score,0.0587,stable (<0.10)
1,pd_hat,0.5869,material shift (>0.25)


**Reading the table:** PSI above 0.25 signals the incoming population has
shifted materially from the reference book -- here, the crisis vintages look
very different from the calm one, which would trigger a model review in
production. This is the kind of monitoring that keeps a deployed model honest.

## Framework alignment notes (APS 113 / APG 113 / Basel / WP14)

These notes record where the build sits against the regulatory framework. The LGD
work (notebooks 01, 04, 04b, 06) now implements economic-loss discounting, the
APRA-view overlays (MI exclusion, 20% reduction, 20% floor), a margin of
conservatism, downturn LGD, a Stage 3 best estimate, and an independent LGD
validation. The remaining items below are **documentation choices**, stated
explicitly so a reviewer can see they were considered.

**Default definition (Step 2 / APS 113 Att D para 5).** This project defines default
at **180 days past due** (status 6+) or a credit-event zero-balance code. 180 DPD is a
common *mortgage* convention and is what the Freddie Mac data cleanly supports. The
APS 220 / Basel reference point is **90 DPD**; we treat the 180-DPD choice as a "broad
equivalence" adjustment and note that a 90-DPD definition would classify more loans as
defaulted earlier (higher PD, generally lower average LGD as more cures are captured).
A **90-DPD sensitivity is already built** (`default_within_12m_90dpd`, from the same
delinquency field) and quantified per grade in notebook 03b -- the one-year rate roughly
doubles under the broader trigger while the rank-ordering holds.

**Cure rules (Step 2).** A loan is counted as defaulted if it *ever* reached 180+ DPD
within its observed history; we do **not** net out subsequent cures in the default flag
(an observed-to-date definition). The resolution-bias analysis in notebook 04 (P2-1)
shows the practical effect: a large share of 180-DPD "defaults" cure or prepay with
little loss, which is why the cure-aware best estimate sits well below the disposed-only
LGD. A production model would add an explicit cure/re-default (probation) window.

**Borrower/collateral correlation (Part 4.1).** In mortgages, falling house prices both
*trigger* defaults (negative equity) and *deepen* losses (smaller recovery on sale), so
PD and LGD are positively correlated in a downturn. We do not model that correlation
parametrically; instead the **downturn LGD** (notebook 04/06) and the joint PD-and-LGD
stress (notebook 07) are the conservative treatment of it -- both drivers worsen together.

**Observation period (Step 7).** The panel now spans **17 origination vintages (2006-2022)**
-- boom-peak, the GFC, the recovery, the long expansion and COVID-2020 -- a **full economic
cycle with two distinct downturns**, observed through 2025-09. This **comfortably exceeds the
framework's 5-year minimum** for PD and retail LGD (APS 113 Att D PD para 4 / LGD para; CRE36.88)
and spans good and bad years as the long-run calibration requires. The **margin of conservatism**
(P2-2) is retained but is now **sized by the data** -- per-grade error bars shrink as observations
accumulate, so the MoC is materially smaller than on the old 3-vintage window while still erring
conservative (CRE36.67). Benchmarking/qualitative review still support the thinnest recent-vintage
LGD cells, whose workouts are not yet fully resolved.

**Segmentation (Step 1).** Current severity drivers are LTV, credit score, loan size
(UPB) and the downturn indicator. In production the LGD segmentation would extend to
**loan purpose, occupancy, and with/without-LMI** (and likely geography/house-price
region), each of which plausibly shifts recovery; they are noted here as the natural
next segments rather than fitted, given the sample size.

## PD framework alignment notes (APS 113 / APG 113 / Basel / WP14)

The PD work (notebooks 01, 03, 03b, 03c) now uses a **one-year default target**
(`default_within_12m`), calibrates each grade to a **count-weighted long-run average**
across vintages, applies a **formal calibration test**, a **risk-sensitive margin of
conservatism**, a **revise-upward ratchet** on under-predicting grades, and the **5 bps
PD floor**. Crucially, this calibrated grade PD now **flows through to Expected Loss
(notebook 06) and the stress test (notebook 07)**, so EL/capital and the stress layer use
the **same PD** as the master scale (EL Part 5.1) -- conservatism reaches the dollar loss,
not just the rating. The items below are the required **documentation** elements.

**Rating philosophy (Step 3 / APG 113 para 73).** This is a **point-in-time-leaning,
through-the-door** scorecard: it uses **origination features only** (credit score, LTV,
DTI, purpose, occupancy, channel) with **no behavioural inputs**, so a loan's grade is
fixed at booking. Through-the-cycle behaviour is *approximated* by calibrating grade PDs
to the **long-run average** of yearly one-year rates (PD-3). We explicitly flag APG 113
para 73's warning that **calibrating PIT ratings to a long-run average does not by itself
make them TTC** -- a genuinely TTC rating would also dampen the rating *migration* through
the cycle, which an origination-only scorecard does not attempt.

**Override policy (Step 8).** No overrides are applied in this demonstration. In
production, rating overrides would be governed by a **written policy** with defined
approval authorities, a documented rationale per override, and **separate tracking and
monitoring** of override rates and their subsequent performance.

**Use test (Part 4.1).** This is a demonstration model and is **not used in live credit
approval, pricing or provisioning**. The "use test" requires that the same ratings drive
real decisions (origination cut-offs, limit-setting, pricing, capital, ECL). Here we show
the *capability* (grades, master scale, EL) but make no claim of operational use.

**Independence (Part 5.8).** Model **development and validation would be independent
functions** in production. In this repo they are separated only by notebook (03/03b build,
03c/04b validate); a real governance setup would place validation in a separate team with
its own sign-off.

**Observation window (Step 5 / PD-8).** Three vintages (2007, 2008, 2015) is **short of a
full cycle**, though it deliberately spans a severe downturn and a calm year -- which is
exactly why the **margin of conservatism** (PD-5) is applied. The framework's one-year
default rate is `(D - E_D) / (N - E_N)` (APG 113 para 110), where `E_D`/`E_N` exclude
zero-exposure and purely technical defaults; in this clean agency sample those exclusions
are **immaterial** (no undrawn/zero-exposure facilities, and defaults are real 180-DPD /
loss events, not technical), so `D/N` is used directly.

**Retail pool framing (Part 2.4 / PD-9).** Residential mortgages are a **retail** asset
class, so the A-H grades are best read as **pools** of homogeneous risk rather than
obligor ratings. A production retail pool system would also separate **delinquent vs
current** exposures and reflect **LGD and EAD pooling**, not PD alone. (The 180-vs-90-DPD
default-definition equivalence under Step 2 is covered in the LGD notes above.)

**Annual review (Part 4.4 / PD-10).** The intended governance cycle is an **annual
revalidation** of the PD (discrimination, calibration test, PSI) and a re-sizing of the
margin of conservatism as more vintages accumulate, plus an out-of-cycle review on a PSI
or calibration-flag trigger.